# Market vs Model Edge Analysis

Compare model fair value with Kalshi midpoint and microprice.

## Finance-Specific EDA: Edge, Baskets, and Relative Value

This section uses the canonical `weather_nyc_main_v1_features` dataset and the corrected contract-aware `forecast_event_indicator` feature. The core quant questions are:

- Are binary books crossed, locked, or systematically tight enough for market making?
- Do mutually exclusive temperature partitions imply executable basket arbitrage?
- Does the weather-implied event direction have enough realized signal to justify taking liquidity?
- Are threshold tails monotone after contract semantics are parsed correctly?

The generated CSVs live under `artifacts/research/` so the same tables can be reused by reports and strategy implementation.

In [ ]:
from pathlib import Path
import polars as pl

RESEARCH = Path("../artifacts/research")
for name in [
    "finance_book_quality_summary",
    "finance_spread_distribution",
    "finance_forecast_edge_bucket_performance",
    "finance_partition_basket_summary",
    "finance_tail_monotonicity_violations",
]:
    path = RESEARCH / f"{name}.csv"
    print(f"\n{name}")
    display(pl.read_csv(path) if path.exists() else pl.DataFrame({"status": [f"missing {path}"]}))

### Trading Interpretation

The book is not crossed at the single-market level, and median spread is 1 cent. That shifts the opportunity away from naive crossed-book arbitrage and toward:

1. **Partition basket arbitrage** across the six mutually exclusive weather contracts for a date.
2. **Relative-value market making** using model fair value, spread, microprice, and inventory limits.
3. **Contract-aware edge taking** only when model edge exceeds fees, spread, and calibration uncertainty.

The partition basket table is the most actionable finance-specific result: when the sum of YES bids exceeds 100 cents, buying the full NO basket is synthetically profitable; when the sum of YES asks is below 100 cents, buying the full YES basket is profitable. These checks should be implemented with one quote per leg per timestamp and strict all-or-none execution assumptions.